In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

df = pd.read_csv('eval/model_vs_labels.csv')
df['match'] = df['label_sentiment'] == df['model_sentiment']
print(f"Total comments: {len(df)}")
print(f"Sentiment matches: {df['match'].sum()} / {len(df)} ({df['match'].mean():.1%})")

In [ ]:
# Confusion matrix
labels = ['very negative', 'negative', 'neutral', 'positive', 'very positive']
present = [l for l in labels if l in df['label_sentiment'].values or l in df['model_sentiment'].values]

cm = confusion_matrix(df['label_sentiment'], df['model_sentiment'], labels=present)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=present, yticklabels=present, ax=ax)
ax.set_xlabel('Model prediction')
ax.set_ylabel('Human label')
ax.set_title('Sentiment — Human vs Model')
plt.tight_layout()
plt.show()

In [ ]:
# Full comparison table — all 58 comments
view = df[['comments', 'label_sentiment', 'model_sentiment', 'match', '_annotator', 'label_tickers']].copy()
view['comments'] = view['comments'].str[:120] + '...'

def color_row(row):
    bg = 'background-color: #d4edda' if row['match'] else 'background-color: #f8d7da'
    return [bg] * len(row)

pd.set_option('display.max_colwidth', 120)
view.style.apply(color_row, axis=1)

In [ ]:
# Mismatches only — full comment text
mismatches = df[~df['match']][['comments', 'label_sentiment', 'model_sentiment', '_annotator', 'label_tickers']].copy()
print(f"{len(mismatches)} mismatches out of {len(df)} comments\n")

for i, row in mismatches.iterrows():
    print(f"--- #{i} | Human: {row['label_sentiment']} | Model: {row['model_sentiment']} | Labeled by: {row['_annotator']} ---")
    print(row['comments'][:500])
    print()

In [ ]:
# Per-annotator accuracy — who agrees most with the model?
per_annotator = df.groupby('_annotator')['match'].agg(['sum', 'count'])
per_annotator['accuracy'] = per_annotator['sum'] / per_annotator['count']
per_annotator.columns = ['correct', 'total', 'accuracy']
per_annotator['accuracy'] = per_annotator['accuracy'].map('{:.1%}'.format)
print(per_annotator.sort_values('correct', ascending=False))